# Pipeline Laminación
Lectura, limpieza y cruce de las dos tablas por hoja (Tren morgan/rollos + Tren 450) con sus parámetros teóricos.

In [1]:
import re
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')

COL_MAP = {
    'CODIGO'                   : 'codigo',
    'material'                 : 'material',
    'MATERIAL'                 : 'material',
    'Cantidad (t)'             : 'cantidad_t',
    'Cantidad (t) Programada'  : 'cantidad_t',       # variante Tren morgan
    'Mtto pro. (h)'            : 'mtto_h',
    'Inicio'                   : 'inicio',
    'Fin'                      : 'fin',
    'Product. (t/h)'           : 'productividad_t_h',
    'Tiempo lam. (h)'          : 'tiempo_lam_h',
    'Índice de utilizacion (%)': 'iu',
    'Índice de utilización (%)': 'iu',
    'Paradas setups (h)'       : 'paradas_setup_h',
    'Paradas por setups (h)'   : 'paradas_setup_h',
    'Paradas imprevistas (h)'  : 'paradas_imprev_h',
    'Tiempo (días)'            : 'tiempo_dias',
    't/día'                    : 't_dia',            # producción diaria Tren morgan
    'ton/dia'                  : 't_dia',            # producción diaria Tren 450
    '%Cumplimiento'            : 'pct_cumplimiento',
}

METRIC_MAP = {
    'capacidade teorica - t/h'                   : 'cap_teorica_t_h',
    'capacidade teorica t/h'                     : 'cap_teorica_t_h',
    'rendimiento'                                : 'rendimiento',
    'eficiencia'                                 : 'eficiencia',
    'produtividad teorica - ton/hora'            : 'prod_teorica_t_h',
    'produtividad teorica - ton/hor'             : 'prod_teorica_t_h',
    'produtividad teorica - ton/h'               : 'prod_teorica_t_h',
    'produtividad teorica - t/h'                 : 'prod_teorica_t_h',
    'iu'                                         : 'iu_teorico',
    'produccion horaria  bruta real'             : 'prod_bruta_t_h',
    'produccion horaria  bruta real  - ton/hora' : 'prod_bruta_t_h',
    'calidad'                                    : 'calidad',
    'produccion horaria  real t/día'             : 'prod_real_t_dia',
    'produccion horaria  real t/dia'             : 'prod_real_t_dia',
}

PROD_PAT  = re.compile(r'N°\d+|[\d.]+mm|[\d]+/[\d]+"')
LABEL_COL = 2
TIPOS     = ['Tren morgan', 'Tren 450']

In [2]:
BRONZE_DIR = Path('../data/bronze')
SILVER_DIR = Path('../data/silver')
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
print("Directorios listos.")

Directorios listos.


In [3]:
def _find_special_rows(raw):
    mask_cod  = raw.apply(lambda r: r.astype(str).str.upper().eq('CODIGO').any(), axis=1)
    mask_prod = raw.apply(
        lambda r: r.astype(str).str.match(PROD_PAT.pattern).sum() >= 3, axis=1
    )
    return raw.index[mask_cod].tolist(), raw.index[mask_prod].tolist()


def leer_hoja(xl, hoja, anio):
    """Extrae y une las dos sub-tablas de datos (Tren morgan + Tren 450)."""
    raw = pd.read_excel(xl, sheet_name=hoja, header=None)
    codigo_rows, _ = _find_special_rows(raw)
    if not codigo_rows:
        return pd.DataFrame()

    bloques = []
    for idx, h_row in enumerate(codigo_rows):
        end_row = codigo_rows[idx + 1] if idx + 1 < len(codigo_rows) else len(raw)
        bloque  = raw.iloc[h_row:end_row].reset_index(drop=True)

        raw_cols = bloque.iloc[0].tolist()
        seen, new_cols = {}, []
        for c in raw_cols:
            key = str(c) if pd.notna(c) else '__nan__'
            seen[key] = seen.get(key, 0) + 1
            new_cols.append(key if seen[key] == 1 else f'{key}_{seen[key]}')
        bloque.columns = new_cols
        bloque = bloque.iloc[1:].reset_index(drop=True)

        col_cod = next((c for c in bloque.columns if c.upper() == 'CODIGO'), None)
        if col_cod:
            bloque = bloque[pd.to_numeric(bloque[col_cod], errors='coerce').notna()].copy()

        rename_map = {c: COL_MAP[c] for c in bloque.columns if c in COL_MAP}
        bloque = bloque.rename(columns=rename_map)
        cols_keep = list(dict.fromkeys(c for c in COL_MAP.values() if c in bloque.columns))
        bloque = bloque[cols_keep].copy()
        bloque['tipo'] = TIPOS[idx] if idx < len(TIPOS) else f'tabla_{idx}'
        bloques.append(bloque)

    df = pd.concat(bloques, ignore_index=True)
    df['anio'] = anio
    df['mes']  = hoja
    return df


def leer_params_hoja(xl, hoja, anio):
    """Extrae parámetros teóricos (cap. teórica, rendimiento, eficiencia…) por producto."""
    raw = pd.read_excel(xl, sheet_name=hoja, header=None)
    codigo_rows, prod_rows = _find_special_rows(raw)
    if not prod_rows:
        return pd.DataFrame()

    frames = []
    for idx, prod_row in enumerate(prod_rows):
        codigo_row = next((c for c in codigo_rows if c > prod_row), None)
        if codigo_row is None:
            continue

        header    = raw.iloc[prod_row]
        prod_cols = {col: str(val) for col, val in header.items()
                     if pd.notna(val) and PROD_PAT.match(str(val))}

        records = []
        for _, row in raw.iloc[prod_row + 1: codigo_row].iterrows():
            label_val = row.iloc[LABEL_COL] if LABEL_COL < len(row) else None
            if pd.isna(label_val) or not isinstance(label_val, str):
                continue
            metrica = METRIC_MAP.get(label_val.strip().lower(), label_val.strip().lower())
            for col, producto in prod_cols.items():
                val = row[col]
                if pd.notna(val):
                    records.append({'producto': producto, 'metrica': metrica, 'valor': float(val)})

        if records:
            df = pd.DataFrame(records)
            df['tipo'] = TIPOS[idx] if idx < len(TIPOS) else f'tabla_{idx}'
            df['anio'] = anio
            df['mes']  = hoja
            frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

## 1. Lectura de órdenes de laminación

In [4]:
frames = []
for archivo in sorted(RAW_DIR.glob('*.xlsx')):
    # Ignorar archivos temporales de Excel (p.ej. ~$...)
    if archivo.name.startswith('~$'):
        print(f"Skipping temporary file: {archivo.name}")
        continue
    anio = int(archivo.stem.split('_')[-1])
    xl   = pd.ExcelFile(archivo, engine='openpyxl')
    for hoja in xl.sheet_names:
        df = leer_hoja(xl, hoja, anio)
        if not df.empty:
            frames.append(df)
        print(f"{archivo.name} / {hoja:<12} → {len(df)} filas")

data = pd.concat(frames, ignore_index=True)
print(f"\nTotal: {data.shape}")
data.head()

programa_laminacion_2025.xlsx / ENERO        → 50 filas
programa_laminacion_2025.xlsx / FEBRERO      → 69 filas
programa_laminacion_2025.xlsx / MARZO        → 69 filas
programa_laminacion_2025.xlsx / ABRIL        → 69 filas
programa_laminacion_2025.xlsx / MAYO         → 73 filas
programa_laminacion_2025.xlsx / JUNIO        → 73 filas
programa_laminacion_2025.xlsx / JULIO        → 73 filas
programa_laminacion_2025.xlsx / AGOSTO       → 73 filas
programa_laminacion_2025.xlsx / SEPTIEMBRE   → 73 filas
programa_laminacion_2025.xlsx / OCTUBRE      → 74 filas
programa_laminacion_2025.xlsx / NOVIEMBRE    → 80 filas
programa_laminacion_2025.xlsx / DICIEMBRE    → 80 filas
programa_laminacion_2026.xlsx / ENERO        → 82 filas
programa_laminacion_2026.xlsx / FEBRERO      → 83 filas
programa_laminacion_2026.xlsx / MARZO        → 83 filas
programa_laminacion_2026.xlsx / ABRIL        → 89 filas

Total: (1193, 17)


,codigo,material,cantidad_t,mtto_h,inicio,fin,productividad_t_h,tiempo_lam_h,iu,paradas_setup_h,paradas_imprev_h,tiempo_dias,t_dia,pct_cumplimiento,tipo,anio,mes
0,6098499,ROLLO CORRU N°3,3459,16,2024-12-31 18:00:00,2025-01-03 15:02:29.904000,69.1125,50.048833,0.724908,4,18.992807,2.876735,1200.0,NaN,Tren morgan,2025,ENERO
1,6098496,ROLLO CORRU 9MM NTC 2289,1095.171791,NaN,2025-01-04 11:02:29.904000,2025-01-05 08:59:00.218000,70.498139,15.534762,0.708,4,6.406992,0.91424,1195.508571,NaN,Tren morgan,2025,ENERO
2,6098498,ROLLO CORRU N°2,2309.708061,16,2025-01-05 12:59:00.218000,2025-01-07 13:05:26.703000,60.29917,38.304144,0.796222,4,9.803214,2.004473,1149.972286,1,Tren morgan,2025,ENERO
3,6098005,"ALAMBRON 5,5mm 10B06",4559.082461,NaN,2025-01-08 09:05:26.703000,2025-01-13 10:24:45.606000,54.428559,83.76269,0.690417,NaN,37.559228,5.05508,900.077624,1,Tren morgan,2025,ENERO
4,6098008,"ALAMBRON 5,5mm ELECTRODO",1465.305893,NaN,2025-01-13 10:24:45.606000,2025-01-15 01:24:21.526000,54.428559,26.921637,0.690417,NaN,12.071674,1.624721,900.077624,1,Tren morgan,2025,ENERO


## 2. Parámetros teóricos por producto

In [5]:
param_frames = []
for archivo in sorted(RAW_DIR.glob('*.xlsx')):
    # Ignorar archivos temporales de Excel (p.ej. ~$...)
    if archivo.name.startswith('~$'):
        print(f"Skipping temporary file: {archivo.name}")
        continue
    anio = int(archivo.stem.split('_')[-1])
    xl   = pd.ExcelFile(archivo, engine='openpyxl')
    for hoja in xl.sheet_names:
        df = leer_params_hoja(xl, hoja, anio)
        if not df.empty:
            param_frames.append(df)

params = pd.concat(param_frames, ignore_index=True)
print(f"Shape params: {params.shape}")
print(f"Métricas : {params['metrica'].unique().tolist()}")
print(f"Productos: {params['producto'].unique().tolist()}")
params.head()

Shape params: (2176, 6)
Métricas : ['cap_teorica_t_h', 'rendimiento', 'eficiencia', 'prod_teorica_t_h', 'iu_teorico', 'prod_bruta_t_h', 'calidad', 'prod_real_t_dia']
Productos: ['N°2', 'N°3', 'N°4', '8.5mm', '9mm', '12mm', '5.5mm', '8mm', '3/8"', '1/4"', '1/2"', 'N°5', 'N°6', 'N°7', 'N°8', 'N°10']


,producto,metrica,valor,tipo,anio,mes
0,N°2,cap_teorica_t_h,70.0,Tren morgan,2025,ENERO
1,N°3,cap_teorica_t_h,75.0,Tren morgan,2025,ENERO
2,N°4,cap_teorica_t_h,75.0,Tren morgan,2025,ENERO
3,8.5mm,cap_teorica_t_h,65.0,Tren morgan,2025,ENERO
4,9mm,cap_teorica_t_h,73.0,Tren morgan,2025,ENERO


## 3. Conversión de tipos

In [6]:
for col in ['cantidad_t', 'mtto_h', 'productividad_t_h', 'tiempo_lam_h',
            'iu', 'paradas_setup_h', 'paradas_imprev_h', 'tiempo_dias']:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')

for col in ['inicio', 'fin']:
    if col in data.columns:
        data[col] = pd.to_datetime(data[col], errors='coerce')

data['codigo'] = data['codigo'].astype(str).str.strip()
print(data.dtypes)

codigo                          str
material                        str
cantidad_t                  float64
mtto_h                      float64
inicio               datetime64[us]
fin                  datetime64[us]
productividad_t_h           float64
tiempo_lam_h                float64
iu                          float64
paradas_setup_h             float64
paradas_imprev_h            float64
tiempo_dias                 float64
t_dia                        object
pct_cumplimiento             object
tipo                            str
anio                          int64
mes                             str
dtype: object


In [7]:
data.to_parquet(BRONZE_DIR / 'laminacion.parquet', index=False)
print(f"Bronze guardado: {data.shape}")
print(f"cantidad_t nulos: {data['cantidad_t'].isna().sum()} / {len(data)} ({data['cantidad_t'].isna().mean()*100:.1f}%)")

Bronze guardado: (1193, 17)
cantidad_t nulos: 453 / 1193 (38.0%)


## 4. Cruce: órdenes + parámetros teóricos

In [8]:
params_wide = params.pivot_table(
    index=['producto', 'tipo', 'anio', 'mes'],
    columns='metrica',
    values='valor',
    aggfunc='first'
).reset_index()
params_wide.columns.name = None
print(params_wide.shape)
params_wide.head()

(272, 12)


,producto,tipo,anio,mes,calidad,cap_teorica_t_h,eficiencia,iu_teorico,prod_bruta_t_h,prod_real_t_dia,prod_teorica_t_h,rendimiento
0,"1/2""",Tren morgan,2025,ABRIL,0.998,65.0,0.9,0.65,36.88425,883.451556,56.745,0.97
1,"1/2""",Tren morgan,2025,AGOSTO,0.998,65.0,0.9,0.65,36.88425,883.451556,56.745,0.97
2,"1/2""",Tren morgan,2025,DICIEMBRE,0.998,65.0,0.9,0.65,36.88425,883.451556,56.745,0.97
3,"1/2""",Tren morgan,2025,ENERO,0.998,65.0,0.9,0.65,36.88425,883.451556,56.745,0.97
4,"1/2""",Tren morgan,2025,FEBRERO,0.998,65.0,0.9,0.65,36.88425,883.451556,56.745,0.97


In [9]:
PROD_EXTRACT = [
    (re.compile(r'N[°.](\d+)'),                    lambda m: f'N°{m.group(1)}'),
    (re.compile(r'(\d+[,.]?\d*)mm', re.I),          lambda m: f'{m.group(1).replace(",", ".")}mm'),
    (re.compile(r'(\d+/\d+)\s*(?:in|pulg)', re.I),  lambda m: f'{m.group(1)}"'),
]

def extraer_producto(material):
    for pat, fmt in PROD_EXTRACT:
        m = pat.search(str(material))
        if m:
            return fmt(m)
    return None

data['producto'] = data['material'].apply(extraer_producto)

data_full = data.merge(params_wide, on=['producto', 'tipo', 'anio', 'mes'], how='left')

print(f"Shape: {data_full.shape}")
print(f"Filas con params: {data_full['cap_teorica_t_h'].notna().sum()} / {len(data_full)}")
data_full.head(10)

Shape: (1193, 26)
Filas con params: 1184 / 1193


,codigo,material,cantidad_t,mtto_h,inicio,fin,productividad_t_h,tiempo_lam_h,iu,paradas_setup_h,...,mes,producto,calidad,cap_teorica_t_h,eficiencia,iu_teorico,prod_bruta_t_h,prod_real_t_dia,prod_teorica_t_h,rendimiento
0,6098499,ROLLO CORRU N°3,3459.000000,16.0,2024-12-31 18:00:00.000,2025-01-03 15:02:29.904,69.112500,50.048833,0.724908,4.0,...,ENERO,N°3,0.998,75.0,0.950000,0.724908,50.100200,1200.000000,69.112500,0.97
1,6098496,ROLLO CORRU 9MM NTC 2289,1095.171791,NaN,2025-01-04 11:02:29.904,2025-01-05 08:59:00.218,70.498139,15.534762,0.708000,4.0,...,ENERO,9mm,0.998,73.0,0.995596,0.708000,49.912683,1195.508571,70.498139,0.97
2,6098498,ROLLO CORRU N°2,2309.708061,16.0,2025-01-05 12:59:00.218,2025-01-07 13:05:26.703,60.299170,38.304144,0.796222,4.0,...,ENERO,N°2,0.998,70.0,0.888058,0.796222,48.011535,1149.972286,60.299170,0.97
3,6098005,"ALAMBRON 5,5mm 10B06",4559.082461,NaN,2025-01-08 09:05:26.703,2025-01-13 10:24:45.606,54.428559,83.762690,0.690417,NaN,...,ENERO,5.5mm,0.998,60.0,0.935199,0.690417,37.578391,900.077624,54.428559,0.97
4,6098008,"ALAMBRON 5,5mm ELECTRODO",1465.305893,NaN,2025-01-13 10:24:45.606,2025-01-15 01:24:21.526,54.428559,26.921637,0.690417,NaN,...,ENERO,5.5mm,0.998,60.0,0.935199,0.690417,37.578391,900.077624,54.428559,0.97
5,6098007,"ALAMBRON 5,5mm 10B22",466.861668,NaN,2025-01-15 01:24:21.526,2025-01-15 13:49:46.751,54.428559,8.577513,0.690417,NaN,...,ENERO,5.5mm,0.998,60.0,0.935199,0.690417,37.578391,900.077624,54.428559,0.97
6,6134124,"ALAMBRON 5,5MM 10B04 NTC 330",650.919040,NaN,2025-01-15 13:49:46.751,2025-01-16 07:09:04.624,54.428559,11.959145,0.690417,NaN,...,ENERO,5.5mm,0.998,60.0,0.935199,0.690417,37.578391,900.077624,54.428559,0.97
7,6097999,"ALAMBRON 5,5MM 1015 NTC 330",361.425070,NaN,2025-01-16 07:09:04.624,2025-01-16 16:46:09.051,54.428559,6.640357,0.690417,NaN,...,ENERO,5.5mm,0.998,60.0,0.935199,0.690417,37.578391,900.077624,54.428559,0.97
8,6097996,"ALAMBRON 5,5mm 1008",497.344964,NaN,2025-01-16 16:46:09.051,2025-01-17 06:00:14.568,54.428559,9.137574,0.690417,NaN,...,ENERO,5.5mm,0.998,60.0,0.935199,0.690417,37.578391,900.077624,54.428559,0.97
9,6097998,"ALAMBRON 5,5mm 1012",514.324413,NaN,2025-01-17 06:00:14.568,2025-01-17 19:41:26.711,54.428559,9.449532,0.690417,4.0,...,ENERO,5.5mm,0.998,60.0,0.935199,0.690417,37.578391,900.077624,54.428559,0.97


In [10]:
MESES_NUM = {
    'ENERO':1,'FEBRERO':2,'MARZO':3,'ABRIL':4,'MAYO':5,'JUNIO':6,
    'JULIO':7,'AGOSTO':8,'SEPTIEMBRE':9,'OCTUBRE':10,'NOVIEMBRE':11,'DICIEMBRE':12
}

silver = data_full.copy()

# Descartar filas sin fecha (no se puede calcular duración)
silver = silver[silver['inicio'].notna() & silver['fin'].notna()].copy()

# Duración real calculada a partir de timestamps
silver['duracion_real_h'] = (silver['fin'] - silver['inicio']).dt.total_seconds() / 3600

# Paradas: NaN → 0 (sin registro = sin parada)
silver['paradas_setup_h']  = silver['paradas_setup_h'].fillna(0)
silver['paradas_imprev_h'] = silver['paradas_imprev_h'].fillna(0)
silver['paradas_total_h']  = silver['paradas_setup_h'] + silver['paradas_imprev_h']

# Tiempo productivo real
silver['tiempo_productivo_h'] = (silver['tiempo_lam_h'] - silver['paradas_total_h']).clip(lower=0)

# Features temporales
silver['hora_inicio'] = silver['inicio'].dt.hour
silver['dia_semana']  = silver['inicio'].dt.dayofweek  # 0=lunes
silver['mes_num']     = silver['mes'].map(MESES_NUM)

# Encoding categórico
silver['tipo_cod'] = silver['tipo'].map({'Tren morgan': 0, 'Tren 450': 1})

# Brecha IU real vs teórico
silver['delta_iu'] = silver['iu'] - silver['iu_teorico']

silver.to_parquet(SILVER_DIR / 'laminacion.parquet', index=False)
print(f"Silver guardado: {silver.shape}")
print(silver[['duracion_real_h','paradas_total_h','tiempo_productivo_h',
              'hora_inicio','dia_semana','mes_num','tipo_cod','delta_iu']].describe().round(2))

Silver guardado: (1192, 34)
       duracion_real_h  paradas_total_h  tiempo_productivo_h  hora_inicio  \
count          1192.00          1192.00              1185.00      1192.00   
mean             15.95             7.02                 5.08        12.77   
std              32.43            13.03                10.82         6.43   
min               0.00             0.00                 0.00         0.00   
25%               0.00             0.00                 0.00         8.00   
50%               0.54             0.52                 0.00        14.00   
75%              13.32             7.40                 3.92        19.00   
max             204.08           112.25                68.60        23.00   

       dia_semana  mes_num  tipo_cod  delta_iu  
count     1192.00  1192.00   1192.00    1183.0  
mean         3.01     5.59      0.39       0.0  
std          1.99     3.51      0.49       0.0  
min          0.00     1.00      0.00       0.0  
25%          1.00     3.00      0

## 6. Gold — capa modelo (XGBoost-ready)

In [11]:
GOLD_DIR = Path('../data/gold')
GOLD_DIR.mkdir(parents=True, exist_ok=True)

DROP_COLS = ['codigo', 'material', 'inicio', 'fin',
             'tipo', 'mes', 'calidad', 'pct_cumplimiento']

gold = silver.drop(columns=[c for c in DROP_COLS if c in silver.columns]).copy()

# Drop 9 rows with no params match (sin parámetros teóricos no se pueden calcular features)
gold = gold.dropna(subset=['cap_teorica_t_h']).reset_index(drop=True)

# Mantenimiento: NaN → 0 (sin registro = sin mtto programado)
gold['mtto_h'] = gold['mtto_h'].fillna(0)

# Cantidad: NaN → mediana por tipo (paradas planificadas sin producción asignada)
gold['cantidad_t'] = gold.groupby('tipo_cod')['cantidad_t'].transform(
    lambda x: x.fillna(x.median())
)

# t_dia y tiempo_dias son object en algunos casos
gold['t_dia']       = pd.to_numeric(gold['t_dia'],       errors='coerce')
gold['tiempo_dias'] = pd.to_numeric(gold['tiempo_dias'], errors='coerce')

for col in ['tiempo_lam_h', 'tiempo_dias', 'tiempo_productivo_h', 't_dia']:
    gold[col] = gold[col].fillna(gold[col].median())

gold.to_parquet(GOLD_DIR / 'laminacion.parquet', index=False)
print(f"Gold guardado: {gold.shape}")
print(f"Columnas ({len(gold.columns)}): {list(gold.columns)}")
remaining_nulls = gold.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0]
print(f"Nulls restantes: {dict(remaining_nulls) if len(remaining_nulls) else 'Ninguno ✓'}")

Gold guardado: (1183, 26)
Columnas (26): ['cantidad_t', 'mtto_h', 'productividad_t_h', 'tiempo_lam_h', 'iu', 'paradas_setup_h', 'paradas_imprev_h', 'tiempo_dias', 't_dia', 'anio', 'producto', 'cap_teorica_t_h', 'eficiencia', 'iu_teorico', 'prod_bruta_t_h', 'prod_real_t_dia', 'prod_teorica_t_h', 'rendimiento', 'duracion_real_h', 'paradas_total_h', 'tiempo_productivo_h', 'hora_inicio', 'dia_semana', 'mes_num', 'tipo_cod', 'delta_iu']
Nulls restantes: Ninguno ✓


## 5. Diagnóstico de calidad

In [12]:
nulos = data_full.isnull().sum().rename('nulos')
pct   = (data_full.isnull().mean() * 100).round(1).rename('%')
print(pd.concat([nulos, pct], axis=1).to_string())
print()
print(data_full.dtypes)
print()
print(data_full.groupby(['anio', 'tipo']).size().rename('filas'))

                   nulos     %
codigo                 0   0.0
material               0   0.0
cantidad_t           453  38.0
mtto_h              1139  95.5
inicio                 1   0.1
fin                    1   0.1
productividad_t_h      0   0.0
tiempo_lam_h           7   0.6
iu                     0   0.0
paradas_setup_h      965  80.9
paradas_imprev_h       1   0.1
tiempo_dias            4   0.3
t_dia                  0   0.0
pct_cumplimiento    1172  98.2
tipo                   0   0.0
anio                   0   0.0
mes                    0   0.0
producto               0   0.0
calidad                9   0.8
cap_teorica_t_h        9   0.8
eficiencia             9   0.8
iu_teorico             9   0.8
prod_bruta_t_h         9   0.8
prod_real_t_dia        9   0.8
prod_teorica_t_h       9   0.8
rendimiento            9   0.8

codigo                          str
material                        str
cantidad_t                  float64
mtto_h                      float64
inicio            